# 01 — Vanilla GISMo (baseline)

The no-health-awareness reference (vanilla GISMo). Its checkpoint is also the prerequisite for the post-hoc filter baseline (`02_filter_baseline.ipynb`).

**Runtime**: Colab free T4 GPU, ~30 min.

**How to run**:
1. Runtime > Change runtime type > T4 GPU
2. Set `PROJECT_ROOT` (cell below) if your Drive layout differs
3. Runtime > Run all

## 1. GPU + dependencies

In [ ]:
!nvidia-smi

In [ ]:
# Versions follow requirements.txt; torch ships with Colab. torch_geometric
# must match the installed torch build — if its import fails later, install
# the PyG wheel for the torch version printed here:
#   https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html
import torch; print('torch', torch.__version__)
!pip install -q 'torch_geometric>=2.4' 'pandas>=2.0' 'numpy>=1.24' matplotlib

## 2. Mount Drive (code + data both live in Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
PROJECT_ROOT = '/content/drive/MyDrive/CS471_project'
CODE_DIR     = f'{PROJECT_ROOT}/code'
DATA_DIR     = f'{PROJECT_ROOT}/data'
OUTPUT_DIR   = f'{PROJECT_ROOT}/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(CODE_DIR)
print(f'CWD        = {os.getcwd()}')
print(f'DATA_DIR   = {DATA_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

In [ ]:
# Fail fast with a clear message if the dataset isn't fully prepared.
# Training needs the substitution pairs + recipes on top of the graph;
# produce them with src/convert_data.py (or notebooks/01_setup_data.ipynb).
_required = ['flavorgraph_edges.csv', 'nodes_filtered.csv', 'usda_mapping.json',
             'pairs_train.csv', 'pairs_val.csv', 'pairs_test.csv', 'recipes.json']
_missing = [f for f in _required if not os.path.exists(f'{DATA_DIR}/{f}')]
if _missing:
    raise FileNotFoundError(
        f'Missing data files in {DATA_DIR}: {_missing}. Prepare the dataset '
        'first (src/convert_data.py or notebooks/01_setup_data.ipynb).')
print('[data] all required files present')

## 3. Smoke test (optional)

In [ ]:
# 1-epoch smoke test (~30 sec on T4). Set RUN_SMOKE = False to skip.
RUN_SMOKE = True
if RUN_SMOKE:
    !python src/train_v1.py --mode baseline --max_epochs 1 --patience 1 --no_resume \
      --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR}/smoke_v1_baseline
    print('\n[smoke] OK')

## 4. Train + test vanilla GISMo

In [ ]:
!python src/train_v1.py --mode baseline \
  --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR}/baseline

print('\n[done] checkpoint:', f'{OUTPUT_DIR}/baseline/best_baseline.pt')

## 5. Test metrics

In [ ]:
import os, json, sys, contextlib, io
import pandas as pd

sys.path.insert(0, f'{CODE_DIR}/src')
from evaluate_health import compute_health_metrics
from evaluate_flavor import evaluate_flavor
from evaluate_id_ood import evaluate_id_ood


def _quiet(fn, *a, **k):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*a, **k)


def collect_metrics(pred_path):
    if not os.path.exists(pred_path):
        return None
    with open(pred_path) as f:
        pred = json.load(f)
    if 'metrics' not in pred:
        print(f'  [skip] {pred_path}: no metrics block')
        return None
    # Tolerate a missing key (older/partial file) — drop just that column.
    out = {k: pred['metrics'][k] for k in ('MRR', 'Hit@1', 'Hit@3', 'Hit@10')
           if k in pred['metrics']}
    try:
        h = _quiet(compute_health_metrics, pred_path,
                   f'{DATA_DIR}/usda_mapping.json', save_to=None)
        out['sugar_sat']   = h['pred']['sugar_g']['satisfaction_rate']
        out['sodium_sat']  = h['pred']['sodium_mg']['satisfaction_rate']
        out['d_sugar_g']   = h['pred']['sugar_g']['avg_delta']
        out['d_sodium_mg'] = h['pred']['sodium_mg']['avg_delta']
    except Exception as e:
        print(f'  [health failed] {e}')
    try:
        fv = _quiet(evaluate_flavor, pred_path,
                    f'{DATA_DIR}/flavorgraph_edges.csv', save_to=None)
        out['flavor_cos'] = fv['pred_cosine']['mean']
    except Exception as e:
        print(f'  [flavor failed] {e}')
    try:
        io_ = _quiet(evaluate_id_ood, pred_path,
                     f'{DATA_DIR}/pairs_train.csv', save_to=None)
        out['ID_MRR']  = io_['id']['MRR']
        out['OOD_MRR'] = io_['ood']['MRR']
    except Exception as e:
        print(f'  [idood failed] {e}')
    return out


def show_table(path_dict):
    rows = []
    for label, path in path_dict.items():
        m = collect_metrics(path)
        if m is None:
            print(f'  skip [{label}]: not found  {path}')
            continue
        rows.append({'config': label, **m})
    if not rows:
        print('(no predictions found)')
        return None
    df = pd.DataFrame(rows).set_index('config')
    cols = ['MRR', 'Hit@1', 'Hit@10', 'sugar_sat', 'sodium_sat',
            'flavor_cos', 'ID_MRR', 'OOD_MRR']
    print(df[[c for c in cols if c in df.columns]].round(2).to_string())
    return df

In [ ]:
print('=== Vanilla GISMo (baseline) — test ===')
_ = show_table({
    'Vanilla GISMo': f'{OUTPUT_DIR}/baseline/test_predictions_baseline.json',
})